In [1]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [2]:
import logging
import pandas as pd
import numpy as np
import awswrangler as wr
from matplotlib import pyplot as plt
import seaborn as sns
from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url
from general_functions.call_api_with_account_id import call_api_with_accountId, send_to_innkeepr_api_paginated

In [3]:
#Lillydoo - 68c2d9007bd2ec4485bb98ed
# Asambeauty - 682ed8362fc068cde38c3dff
# Störtebekker - 69fc8fed886052eb05c43ac5
# Junglueck - 682b1d8a1de6c855565aeff2
# Plantura - 
conversion_action_id = "69fc8fed886052eb05c43ac5" #Nikin: 6834787b13526dc3d1017e06"
customer = "Störtebekker"
start_date = "20260529"
end_date = "20260529"
is_test = False
test_path = ""
date_range = pd.date_range(start=start_date, end=end_date, freq="D").strftime("%Y%m%d").tolist()
url = return_api_url()
print(f"url = {url}")
account_id = return_workspace_ids()
account_id = [acc["id"] for acc in account_id if acc["name"] == customer]
account_id = account_id[0]

# Load Data

In [4]:
data_file_path = f"DataChecks/targeting_history_ga_conversion_update/analyze/targeting_history_{customer}_{conversion_action_id}_{start_date}_{end_date}_{is_test}.csv"
try:
    if is_test:
        data_file_path = f"{data_file_path}-xxx"
    df = pd.read_csv(data_file_path)
except FileNotFoundError:
    print("File not found, creating new DataFrame.")
    df = pd.DataFrame()
    for date in date_range:
        try:
            print(f"Reading data for {date}")
            if is_test:
                path = test_path
            else:
                path = f"s3://{account_id}/targeting.history/{date}/ga_conversion_update_{conversion_action_id}.parquet"
            temp = wr.s3.read_parquet(path)
        except wr.exceptions.NoFilesFound:
            print(f". No data for {date}")
            continue
        temp["bucket_date"] = date
        df = pd.concat([df, temp])
        if is_test:
            break
    df.to_csv(data_file_path, index=False)

In [5]:
df = df.rename(columns={"bucket_date": "upload_date"})
df = df[["upload_date","anonymousId","profile","session","session.date","traffic_type","properties.revenue", "final_adjusted_revenue"]]

In [6]:
df.to_csv(f"/Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/DataChecks/targeting_history_ga_conversion_update/analyze/data_{customer}_{conversion_action_id}_{start_date}_{end_date}.csv", index=False)

# Number daily conversion per day by traffic type

The number of daily conversions we track (via conversions/query)

In [7]:
df.groupby(by=["session.date"])["session"].count().plot(kind="bar")
plt.grid(True)
plt.title("Count Daily Conversions")

# Analyze by Traffic Type

Count Daily Conversions by traffic type
- Number of daily conversions splitted by traffic type

In [8]:
count_conv_by_traffic_type = df.groupby(by=["session.date","traffic_type"])["session"].count().reset_index()
fig = plt.figure(figsize=(8,4))
ax = fig.add_subplot(111)
sns.lineplot(data=count_conv_by_traffic_type, x="session.date", y="session", hue="traffic_type", ax=ax, errorbar=None)
plt.grid(True)
plt.title("Count Daily Conversions by traffic type")
plt.xticks(rotation=90)
plt.show()

The original revenue and the adjusted revenue modified by the bidding logic for one upload date with outlook=30 splitted by session date.

In [23]:
conv_sum_by_traffic_type = df.groupby(by=["session.date","traffic_type"])["properties.revenue"].sum().reset_index()
adj_conv_sum_by_traffic_type = df.groupby(by=["session.date","traffic_type"])["final_adjusted_revenue"].sum().reset_index()

fig = plt.figure(figsize=(8,4))
fig.suptitle(f"{customer}: brand vs. generic revenue change (-- orignal revenue)")
ax = fig.add_subplot(111)

sns.lineplot(data=conv_sum_by_traffic_type, x="session.date", y="properties.revenue", hue="traffic_type", linestyle="--", ax=ax, errorbar=None, legend=False)
sns.lineplot(data=adj_conv_sum_by_traffic_type, x="session.date", y="final_adjusted_revenue", hue="traffic_type", ax=ax, errorbar=None, legend=False)

traffic_types = conv_sum_by_traffic_type["traffic_type"].unique()
label_map = {"brand": "Brand", "generic": "Generic"}
n = len(traffic_types)

handles = ax.get_lines()[:n]
labels = [label_map.get(tt, tt) for tt in traffic_types]
ax.legend(handles, labels)
ax.set_ylim([0,35000])

plt.grid(True)
plt.xticks(rotation=90)
plt.show()

# Difference in Percentage between uploaded revenue and original revenue

In [10]:
# percentage
sum_revenue = df.groupby(by=["traffic_type"])[["final_adjusted_revenue","properties.revenue"]].sum()
sum_revenue["percentage"] = sum_revenue["final_adjusted_revenue"] / sum_revenue["properties.revenue"] * 100
sum_revenue = sum_revenue.rename(columns={"final_adjusted_revenue": "uploaded_revenue", "properties.revenue": "original revenue"})
sum_revenue

In [11]:
sum_revenue_daily = df.groupby(by=["session.date","traffic_type"])[["final_adjusted_revenue","properties.revenue"]].sum()
sum_revenue_daily["percentage"] = sum_revenue_daily["final_adjusted_revenue"] / sum_revenue_daily["properties.revenue"] * 100
sum_revesum_revenue_dailynue = sum_revenue_daily.rename(columns={"final_adjusted_revenue": "uploaded_revenue", "properties.revenue": "original revenue"})
sum_revenue_daily.sort_values(by=["session.date","traffic_type"])
sum_revenue_daily

In [12]:
fig = plt.figure(figsize=(8,4))
ax = fig.add_subplot(111)
sns.lineplot(data=sum_revenue_daily, x="session.date", y="percentage", hue="traffic_type", ax=ax, errorbar=None)
plt.grid(True)
plt.title("Percentage (uploaded revenue / original revenue * 100) over time")
plt.xticks(rotation=90)
plt.show()

In [13]:
!jupyter nbconvert --to webpdf DataChecks/targeting_history_ga_conversion_update/analyze_history.ipynb

In [14]:
!jupyter nbconvert --to html DataChecks/targeting_history_ga_conversion_update/analyze_history.ipynb